# Notebook 05 — Build Spatial Baselines from Visium Data

**Purpose**: Extend the Healthy Respiratory Model with spatial context by computing
gene expression baselines for each anatomical region of healthy lung tissue.

The deviation engine currently answers: *what is biologically wrong?*
After this notebook it also answers: *where in the lung tissue is it wrong?*

---

## Why spatial baselines matter

The same gene can be completely normal in one region of the lung and severely
abnormal in another. Without spatial context:

- COL1A1 elevated globally → `"fibrotic signal detected"`
- COL1A1 elevated in parenchyma, normal in bronchus → `"subpleural/alveolar fibrosis — not airway"`

The second answer is the one a clinician can act on.

---

## Data used

| File | Spots | Tissue | Source |
|---|---|---|---|
| `lung_tissues_visium_bronchus.h5ad` | 4,992 | Adult bronchus | CellxGene Spatial |
| `lung_tissues_visium_parenchyma.h5ad` | 4,992 | Adult parenchyma | CellxGene Spatial |
| `lung_fetal_visium_annotated.h5ad` | 22,883 | Fetal lung (10 domains) | Sanger Spatial |

Each Visium spot is a 55 µm circle in tissue that captures expression from
approximately 1–10 cells. Unlike scRNA-seq, each measurement has a known
physical location in the tissue section.

---

## What this notebook produces

`models/respiratory_model/spatial_baselines.json` — loaded by the API at startup.

Structure:
```
{
  "bronchus": {
    "overall": { gene → {mean, std, p5, p95} },
    "regions": { region_name → { gene → {mean, std, p5, p95} } },
    "spatially_variable_genes": [ top genes that vary across regions ],
    "region_markers": { region → top marker genes },
    "n_spots": int
  },
  "parenchyma": { ... },
  "fetal": { ... },
  "built_at": "ISO timestamp"
}
```

After this notebook runs, `GET /api/v1/model/status` will return `has_spatial_baselines: true`.

---
## Section 1 — Setup

In [ ]:
import subprocess, sys

def pip_install(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *pkgs])

# Core single-cell stack
try:
    import anndata
    import scanpy as sc
    print(f'scanpy {sc.__version__}  anndata {anndata.__version__}  — ready.')
except ImportError:
    pip_install('anndata>=0.10.0', 'scanpy>=1.10.0')
    print('Installed scanpy + anndata.')

# Spatial analysis — squidpy for Moran's I and spatial clustering
try:
    import squidpy as sq
    print(f'squidpy {sq.__version__} — ready.')
except ImportError:
    pip_install('squidpy>=1.4.0')
    import squidpy as sq
    print('Installed squidpy.')

In [ ]:
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import anndata as ad
import numpy as np
import pandas as pd
import scanpy as sc
import squidpy as sq
from scipy.sparse import issparse
from scipy import stats as scipy_stats

sc.settings.verbosity = 1
print('All imports OK.')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Paths — adjust if your Drive layout differs ─────────────────────────────
from pathlib import Path

DRIVE_ROOT  = Path('/content/drive/MyDrive')
DATA_DIR    = DRIVE_ROOT / 'HEALTH' / 'data'
MODEL_DIR   = DRIVE_ROOT / 'HEALTH' / 'models' / 'respiratory_model'

BRONCHUS_FILE   = DATA_DIR / 'lung_tissues_visium_bronchus.h5ad'
PARENCHYMA_FILE = DATA_DIR / 'lung_tissues_visium_parenchyma.h5ad'
FETAL_FILE      = DATA_DIR / 'lung_fetal_visium_annotated.h5ad'
# ────────────────────────────────────────────────────────────────────────────

print(f'Drive root: {DRIVE_ROOT}')
print(f'Data dir:   {DATA_DIR}')
print(f'Model dir:  {MODEL_DIR}')
print()
if not MODEL_DIR.exists():
    raise RuntimeError('Model artefacts not found. Run Notebook 02 first.')
print('Model artefacts found. Ready to download and process Visium data.')


In [ ]:
# ── Download Visium files to Drive if not already there ─────────────────────
import os, time, requests
from tqdm import tqdm

VISIUM_FILES = {
    BRONCHUS_FILE: {
        'url':  'https://datasets.cellxgene.cziscience.com/6c241a66-d15d-4357-bdbe-7274fdbe20c9.h5ad',
        'name': 'Visium Bronchus',
        'size': '695 MB',
    },
    PARENCHYMA_FILE: {
        'url':  'https://datasets.cellxgene.cziscience.com/a13eb499-f3ae-4c9b-b9ce-89fbddd17d53.h5ad',
        'name': 'Visium Parenchyma',
        'size': '831 MB',
    },
    FETAL_FILE: {
        'url':  'https://cellgeni.cog.sanger.ac.uk/fetal-lung/visium/220627AnnotatedLungVisium.h5ad',
        'name': 'Visium Fetal (optional)',
        'size': '1.45 GB',
    },
}

DATA_DIR.mkdir(parents=True, exist_ok=True)

def download_with_resume(url, dest, name, retries=5):
    for attempt in range(retries):
        try:
            headers = {}
            mode = 'wb'
            if dest.exists():
                existing = dest.stat().st_size
                headers['Range'] = f'bytes={existing}-'
                mode = 'ab'
                print(f'  Resuming from {existing/1e6:.1f} MB...')
            r = requests.get(url, headers=headers, stream=True, timeout=60)
            r.raise_for_status()
            total = int(r.headers.get('content-length', 0))
            if 'Range' in headers:
                total += existing
            with open(dest, mode) as f, tqdm(
                desc=name, total=total, unit='iB',
                unit_scale=True, unit_divisor=1024
            ) as bar:
                if 'Range' in headers:
                    bar.update(existing)
                for chunk in r.iter_content(chunk_size=1024*1024):
                    if chunk:
                        f.write(chunk)
                        bar.update(len(chunk))
            return True
        except Exception as e:
            print(f'  Attempt {attempt+1} failed: {e}')
            time.sleep(5)
    return False

print('Checking Visium files in Drive...\n')
for path, info in VISIUM_FILES.items():
    if path.exists():
        size_mb = path.stat().st_size / 1e6
        print(f'  ✓ {info["name"]} already in Drive ({size_mb:.0f} MB)')
    else:
        print(f'  ↓ Downloading {info["name"]} ({info["size"]})...')
        ok = download_with_resume(info['url'], path, info['name'])
        if ok:
            print(f'  ✓ {info["name"]} saved to Drive')
        else:
            if 'optional' in info['name'].lower():
                print(f'  ⚠ {info["name"]} failed — will be skipped (not required)')
            else:
                raise RuntimeError(f'Failed to download {info["name"]}. Check your internet connection and try again.')

print()
print('All required files ready.')
# Verify again
for name, path in [
    ('Bronchus',   BRONCHUS_FILE),
    ('Parenchyma', PARENCHYMA_FILE),
    ('Fetal',      FETAL_FILE),
]:
    status = '✓' if path.exists() else '✗ missing'
    size   = f'{path.stat().st_size/1e6:.0f} MB' if path.exists() else ''
    print(f'  {status}  {name}: {size}')

In [ ]:
# Verify all files are in place before proceeding
missing = []
for name, path in [
    ('Bronchus Visium',   BRONCHUS_FILE),
    ('Parenchyma Visium', PARENCHYMA_FILE),
    ('Model dir',         MODEL_DIR),
]:
    status = '✓' if path.exists() else '✗ NOT FOUND'
    print(f'  {status}  {name}: {path}')
    if not path.exists() and name != 'Fetal Visium':
        missing.append(name)

if not MODEL_DIR.exists():
    raise RuntimeError('Model artefacts not found. Run Notebook 02 first.')
if missing:
    raise RuntimeError(f'Still missing after download attempt: {missing}. Re-run the download cell.')

print(f'\nFetal Visium: {"found" if FETAL_FILE.exists() else "not found — will be skipped"}')
print('\nAll required files ready. Proceeding...')


---
## Section 2 — Load Existing Model Artefacts

We need the gene list from Notebook 02 so we only compute spatial baselines
for genes the engine already tracks. No point building spatial context for
genes the deviation engine cannot score.

In [ ]:
def load_json(name):
    return json.loads((MODEL_DIR / name).read_text())

META        = load_json('meta.json')
GENE_STATS  = load_json('gene_stats.json')
TRACKED_GENES = set(GENE_STATS.keys())

print(f'Model built at:  {META["built_at"]}')
print(f'Tracked genes:   {len(TRACKED_GENES):,}')
print(f'Healthy cells:   {META["n_cells"]:,}')
print(f'Spatial already: {META.get("has_spatial_baselines", False)}')


# Diagnose: what format are TRACKED_GENES in?
sample_keys = sorted(TRACKED_GENES)[:5]
print(f'\nTracked gene sample (first 5): {sample_keys}')
n_ensembl = sum(1 for g in TRACKED_GENES if str(g).startswith('ENSG'))
print(f'Ensembl IDs in tracked genes: {n_ensembl}/{len(TRACKED_GENES)} ({n_ensembl/len(TRACKED_GENES)*100:.0f}%)')
if n_ensembl > len(TRACKED_GENES) * 0.5:
    print('  → gene_stats.json uses ENSEMBL IDs')
else:
    print('  → gene_stats.json uses GENE SYMBOLS')

---
## Section 3 — Helper Functions

In [ ]:
def to_dense(X):
    """Convert sparse or array slice to a flat numpy float32 array."""
    if issparse(X):
        return np.asarray(X.todense(), dtype=np.float32)
    return np.asarray(X, dtype=np.float32)


def normalise_to_log1p_cp10k(adata):
    """
    Normalise a Visium AnnData object to log1p CP10K in place.
    Skips normalisation if the data already appears to be log-normalised
    (max value < 20 — raw counts would be in the thousands).
    """
    X_sample = to_dense(adata.X[:10, :100])
    max_val   = float(X_sample.max())

    if max_val < 20:
        print(f'  Data appears already log-normalised (max sample value: {max_val:.2f}). Skipping.')
        return

    print(f'  Normalising from raw counts (max sample value: {max_val:.0f})...')
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    print(f'  Done. New max sample value: {float(to_dense(adata.X[:10, :100]).max()):.2f}')


def gene_baseline(matrix_rows, gene_names, tracked):
    """
    Given a 2D float32 matrix (spots × genes) and corresponding gene names,
    compute {gene: {mean, std, p5, p95}} for all genes in `tracked`.
    """
    baseline = {}
    for i, gene in enumerate(gene_names):
        if gene not in tracked:
            continue
        col = matrix_rows[:, i]
        baseline[gene] = {
            'mean': round(float(col.mean()), 4),
            'std':  round(float(col.std()),  4),
            'p5':   round(float(np.percentile(col,  5)), 4),
            'p95':  round(float(np.percentile(col, 95)), 4),
        }
    return baseline


def top_marker_genes(region_matrix, other_matrix, gene_names, tracked, top_n=20):
    """
    Find genes most enriched in `region_matrix` compared to `other_matrix`.
    Uses mean fold-change on log-normalised data.
    Returns list of (gene, fold_change) tuples.
    """
    markers = []
    for i, gene in enumerate(gene_names):
        if gene not in tracked:
            continue
        region_mean = float(region_matrix[:, i].mean())
        other_mean  = float(other_matrix[:, i].mean())
        fc = region_mean - other_mean  # log-space → log fold change
        if fc > 0.3:  # enriched by at least 0.3 log1p units
            markers.append((gene, round(fc, 3)))
    markers.sort(key=lambda x: x[1], reverse=True)
    return markers[:top_n]


def spatially_variable_genes(adata, gene_names, tracked, top_n=100):
    """
    Identify genes with high spatial variance using Moran's I.
    Falls back to top-variance genes if squidpy fails.
    """
    try:
        sq.gr.spatial_neighbors(adata, coord_type='generic', delaunay=True)
        sq.gr.spatial_autocorr(adata, mode='moran', n_perms=100, n_jobs=1)
        morans = adata.uns['moranI'].sort_values('I', ascending=False)
        svgs = [g for g in morans.index if g in tracked][:top_n]
        print(f'  Moran\'s I: found {len(svgs)} spatially variable genes (top Moran I)')
        return svgs
    except Exception as e:
        print(f'  Moran\'s I failed ({e}). Falling back to high-variance genes.')
        gene_idx  = [i for i, g in enumerate(gene_names) if g in tracked]
        X_tracked = to_dense(adata.X)[:, gene_idx]
        variances = X_tracked.var(axis=0)
        top_idx   = np.argsort(variances)[::-1][:top_n]
        svgs = [gene_names[gene_idx[i]] for i in top_idx]
        print(f'  Variance fallback: {len(svgs)} high-variance genes.')
        return svgs




def get_gene_names(adata):
    """
    Return gene symbols for an AnnData object.
    CellxGene h5ad files store Ensembl IDs in var_names and symbols in
    var['feature_name']. This function prefers symbols so TRACKED_GENES matches.
    """
    # Try common symbol columns used by CellxGene / 10x exports
    for col in ['feature_name', 'gene_name', 'gene_symbol', 'symbol', 'feature_id']:
        if col in adata.var.columns:
            names = adata.var[col].tolist()
            n_ensembl = sum(1 for n in names[:200] if str(n).startswith('ENSG'))
            if n_ensembl < 100:  # mostly symbols
                print(f'  Gene names from var["{col}"] — {len(names):,} genes (symbols)')
                return names
    # Fallback to var_names (may be Ensembl IDs)
    names = list(adata.var_names)
    n_ensembl = sum(1 for n in names[:200] if str(n).startswith('ENSG'))
    if n_ensembl > 100:
        print(f'  ⚠  var_names look like Ensembl IDs — checking for symbol columns...')
        print(f'     var columns: {list(adata.var.columns)}')
        print(f'     Sample var_names: {names[:5]}')
    else:
        print(f'  Gene names from var_names — {len(names):,} genes')
    return names


print('Helper functions defined.')

---
## Section 4 — Process Bronchus Visium Data

Adult bronchus tissue — the airway. Contains epithelial layers, submucosa,
cartilage, and glands. Gene expression here reflects airway biology:
mucus production, mucociliary clearance, airway defence.

In [ ]:
print('Loading bronchus Visium data...')
t0 = time.time()
bronchus = ad.read_h5ad(BRONCHUS_FILE, backed='r').to_memory()
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'  Shape:      {bronchus.shape}  (spots × genes)')
print(f'  Obs columns: {list(bronchus.obs.columns)}')
print(f'  ObsM keys:   {list(bronchus.obsm.keys())}')
print()
print(bronchus.obs.head(3))

In [ ]:
# Filter to healthy tissue only (disease == normal)
if 'disease' in bronchus.obs.columns:
    print('Disease values:', bronchus.obs['disease'].value_counts().to_dict())
    bronchus = bronchus[bronchus.obs['disease'] == 'normal'].copy()
    print(f'After filtering to normal: {bronchus.n_obs} spots')
else:
    print('No disease column — assuming all spots are healthy (Visium tissue sections).')

print(f'Spots for baseline: {bronchus.n_obs}')

In [ ]:
# Normalise if needed
print('Checking normalisation...')
normalise_to_log1p_cp10k(bronchus)

In [ ]:
# Detect tissue region annotations
# Common column names in CellxGene Visium exports
REGION_CANDIDATES = [
    # Exact column names found in CellxGene + Sanger Visium exports
    'regions', 'annotation', 'region',                        # fetal Sanger + generic
    'tissue_section', 'tissue_level_1', 'tissue_level_2', 'tissue_level_3',
    'layer', 'domain', 'spatial_cluster', 'clusters', 'cluster', 'leiden_R',
    'ann_level_1', 'ann_level_2', 'cell_type',
]

region_col = None
for col in REGION_CANDIDATES:
    if col in bronchus.obs.columns:
        n_unique = bronchus.obs[col].nunique()
        print(f'  Found candidate column: "{col}" — {n_unique} unique values')
        if 1 < n_unique <= 30:  # meaningful but not too granular
            region_col = col
            print(f'  → Using "{col}" as region column')
            print(f'    Values: {bronchus.obs[col].value_counts().to_dict()}')
            break

if region_col is None:
    print('No region annotation found. Computing spatial clusters via Leiden...')

In [ ]:
# If no region annotations — compute spatial clusters
if region_col is None:
    print('Computing spatial clusters from gene expression + spatial coordinates...')

    # PCA on tracked genes only
    gene_names_b = get_gene_names(bronchus)
    tracked_idx  = [i for i, g in enumerate(gene_names_b) if g in TRACKED_GENES]

    # Create a small AnnData with only tracked genes for clustering
    bronchus_sub = bronchus[:, tracked_idx].copy()

    sc.pp.pca(bronchus_sub, n_comps=30, svd_solver='arpack')

    # Use spatial coordinates to inform clustering if available
    if 'spatial' in bronchus.obsm:
        sq.gr.spatial_neighbors(bronchus_sub, coord_type='generic', delaunay=True)
        sc.pp.neighbors(bronchus_sub, use_rep='X_pca', n_neighbors=15)
    else:
        sc.pp.neighbors(bronchus_sub, use_rep='X_pca', n_neighbors=15)

    sc.tl.leiden(bronchus_sub, resolution=0.4, random_state=42)
    bronchus.obs['spatial_region'] = bronchus_sub.obs['leiden'].values
    region_col = 'spatial_region'

    n_clusters = bronchus.obs[region_col].nunique()
    print(f'  Found {n_clusters} spatial clusters')
    print(f'  Cluster sizes: {bronchus.obs[region_col].value_counts().to_dict()}')
else:
    print(f'Using existing annotation column: "{region_col}"')

In [ ]:
# Build bronchus baselines
print('Computing bronchus baselines...')
gene_names_b  = get_gene_names(bronchus)
X_bronchus    = to_dense(bronchus.X)
regions_b     = bronchus.obs[region_col].astype(str).values

# Overall bronchus baseline (all spots)
print(f'  Overall baseline ({bronchus.n_obs} spots)...')
overall_b = gene_baseline(X_bronchus, gene_names_b, TRACKED_GENES)
print(f'  → {len(overall_b)} genes')

# Per-region baselines
region_baselines_b = {}
region_markers_b   = {}
unique_regions     = sorted(set(regions_b))

for region in unique_regions:
    mask        = regions_b == region
    X_region    = X_bronchus[mask]
    X_other     = X_bronchus[~mask]
    n_spots     = int(mask.sum())
    print(f'  Region "{region}": {n_spots} spots...')

    region_baselines_b[region] = {
        'n_spots':        n_spots,
        'gene_baselines': gene_baseline(X_region, gene_names_b, TRACKED_GENES),
    }

    # Top marker genes for this region vs all others
    markers = top_marker_genes(X_region, X_other, gene_names_b, TRACKED_GENES, top_n=20)
    region_markers_b[region] = [{'gene': g, 'log_fc': fc} for g, fc in markers]
    if markers:
        print(f'    Top markers: {[g for g, _ in markers[:5]]}')

print('\nComputing spatially variable genes (Moran\'s I)...')
svgs_b = spatially_variable_genes(bronchus, gene_names_b, TRACKED_GENES, top_n=100)

BRONCHUS_RESULT = {
    'n_spots':                   int(bronchus.n_obs),
    'region_column_used':        region_col,
    'n_regions':                 len(unique_regions),
    'region_names':              unique_regions,
    'overall':                   overall_b,
    'regions':                   region_baselines_b,
    'region_markers':            region_markers_b,
    'spatially_variable_genes':  svgs_b,
}

print(f'\nBronchus done: {len(overall_b)} gene baselines, {len(unique_regions)} regions, {len(svgs_b)} SVGs')

---
## Section 5 — Process Parenchyma Visium Data

Adult lung parenchyma — the deep tissue. Contains alveoli, alveolar macrophages,
capillaries, and AT1/AT2 cells. This is where gas exchange happens and where most
fibrotic and infectious disease begins.

In [ ]:
print('Loading parenchyma Visium data...')
t0 = time.time()
parenchyma = ad.read_h5ad(PARENCHYMA_FILE, backed='r').to_memory()
print(f'Loaded in {time.time()-t0:.1f}s')
print(f'  Shape:       {parenchyma.shape}')
print(f'  Obs columns: {list(parenchyma.obs.columns)}')
print()
print(parenchyma.obs.head(3))

In [ ]:
# Filter to healthy
if 'disease' in parenchyma.obs.columns:
    print('Disease values:', parenchyma.obs['disease'].value_counts().to_dict())
    parenchyma = parenchyma[parenchyma.obs['disease'] == 'normal'].copy()
    print(f'After filtering: {parenchyma.n_obs} spots')
else:
    print('No disease column — assuming healthy tissue.')

# Normalise
normalise_to_log1p_cp10k(parenchyma)

In [ ]:
# Detect or compute region annotations
region_col_p = None
for col in REGION_CANDIDATES:
    if col in parenchyma.obs.columns:
        n_unique = parenchyma.obs[col].nunique()
        print(f'  Found candidate: "{col}" — {n_unique} values')
        if 1 < n_unique <= 30:
            region_col_p = col
            print(f'  → Using "{col}"')
            print(f'    Values: {parenchyma.obs[col].value_counts().to_dict()}')
            break

if region_col_p is None:
    print('No annotation found. Computing spatial clusters...')
    gene_names_p = get_gene_names(parenchyma)
    tracked_idx_p = [i for i, g in enumerate(gene_names_p) if g in TRACKED_GENES]
    para_sub = parenchyma[:, tracked_idx_p].copy()
    sc.pp.pca(para_sub, n_comps=30, svd_solver='arpack')
    if 'spatial' in parenchyma.obsm:
        sq.gr.spatial_neighbors(para_sub, coord_type='generic', delaunay=True)
    sc.pp.neighbors(para_sub, use_rep='X_pca', n_neighbors=15)
    sc.tl.leiden(para_sub, resolution=0.4, random_state=42)
    parenchyma.obs['spatial_region'] = para_sub.obs['leiden'].values
    region_col_p = 'spatial_region'
    print(f'  {parenchyma.obs[region_col_p].nunique()} clusters found')
else:
    print(f'Using annotation: "{region_col_p}"')

In [ ]:
# Build parenchyma baselines
print('Computing parenchyma baselines...')
gene_names_p  = get_gene_names(parenchyma)
X_parenchyma  = to_dense(parenchyma.X)
regions_p     = parenchyma.obs[region_col_p].astype(str).values
unique_p      = sorted(set(regions_p))

overall_p = gene_baseline(X_parenchyma, gene_names_p, TRACKED_GENES)
print(f'  Overall: {len(overall_p)} genes')

region_baselines_p = {}
region_markers_p   = {}

for region in unique_p:
    mask     = regions_p == region
    X_region = X_parenchyma[mask]
    X_other  = X_parenchyma[~mask]
    n_spots  = int(mask.sum())
    print(f'  Region "{region}": {n_spots} spots')

    region_baselines_p[region] = {
        'n_spots':        n_spots,
        'gene_baselines': gene_baseline(X_region, gene_names_p, TRACKED_GENES),
    }
    markers = top_marker_genes(X_region, X_other, gene_names_p, TRACKED_GENES, top_n=20)
    region_markers_p[region] = [{'gene': g, 'log_fc': fc} for g, fc in markers]
    if markers:
        print(f'    Top markers: {[g for g, _ in markers[:5]]}')

print('\nComputing parenchyma SVGs...')
svgs_p = spatially_variable_genes(parenchyma, gene_names_p, TRACKED_GENES, top_n=100)

PARENCHYMA_RESULT = {
    'n_spots':                   int(parenchyma.n_obs),
    'region_column_used':        region_col_p,
    'n_regions':                 len(unique_p),
    'region_names':              unique_p,
    'overall':                   overall_p,
    'regions':                   region_baselines_p,
    'region_markers':            region_markers_p,
    'spatially_variable_genes':  svgs_p,
}

print(f'\nParenchyma done: {len(overall_p)} gene baselines, {len(unique_p)} regions, {len(svgs_p)} SVGs')

---
## Section 6 — Process Fetal Visium Data (Optional)

22,883 spots across 10 annotated developmental domains. This gives a spatial
picture of where fetal lung development programmes are active — useful for
detecting fetal programme reactivation in adult disease (e.g. dedifferentiation
in fibrosis or cancer reactivates developmental pathways in specific regions).

This section is skipped gracefully if the file is not in Drive.

In [ ]:
# ── Free memory before loading fetal data ────────────────────────────────────
# Bronchus and parenchyma matrices are large. Release them before loading fetal.
import gc

# Keep only the computed results — release the raw matrices
try:
    del bronchus
    print('Released bronchus matrix')
except NameError:
    pass
try:
    del parenchyma
    print('Released parenchyma matrix')
except NameError:
    pass
try:
    del X_bronchus, X_parenchyma
    print('Released dense matrices')
except NameError:
    pass

gc.collect()

# Check available memory
import psutil
mem = psutil.virtual_memory()
print(f'RAM available: {mem.available / 1e9:.1f} GB / {mem.total / 1e9:.1f} GB total')
print('Ready to load fetal data.')


In [ ]:
FETAL_RESULT = None

if FETAL_FILE.exists():
    print('Loading fetal Visium data (backed mode — memory efficient)...')
    t0 = time.time()

    # Keep backed='r' — do NOT call .to_memory() on this large file
    fetal_vis = ad.read_h5ad(FETAL_FILE, backed='r')
    print(f'Loaded in {time.time()-t0:.1f}s  —  {fetal_vis.shape}')

    # ── Find region column ────────────────────────────────────────────────────
    region_col_f = None
    for col in REGION_CANDIDATES:
        if col in fetal_vis.obs.columns:
            n_u = fetal_vis.obs[col].nunique()
            if 1 < n_u <= 50:
                region_col_f = col
                print(f'  Region column: "{col}" — {n_u} values')
                print(f'  Regions: {sorted(fetal_vis.obs[col].astype(str).unique())}')
                break

    if region_col_f is None:
        region_col_f = 'leiden_R' if 'leiden_R' in fetal_vis.obs.columns else '__all__'
        if region_col_f == '__all__':
            fetal_vis.obs['__all__'] = 'fetal_lung'
        print(f'  Using: {region_col_f}')

    # ── Cell type deconvolution columns ───────────────────────────────────────
    q05_cols = [c for c in fetal_vis.obs.columns if c.startswith('q05cell_abundance')]
    slt_cols = [c for c in fetal_vis.obs.columns if c.startswith('SLT: ')]
    print(f'  SLT scores: {len(slt_cols)} | q05 abundance: {len(q05_cols)}')

    gene_names_f = get_gene_names(fetal_vis)
    regions_f    = fetal_vis.obs[region_col_f].astype(str).values
    unique_f     = sorted(set(regions_f))

    print(f'\nProcessing {len(unique_f)} regions one at a time (memory-safe)...')

    # ── Process one region at a time — never load full matrix ────────────────
    overall_counts  = {}   # gene -> running sum and count for overall baseline
    overall_sq      = {}   # gene -> sum of squares for std
    overall_n       = 0
    overall_vals    = {}   # for percentiles: gene -> list (sampled)

    region_baselines_f   = {}
    ct_abundance_per_region = {}

    CHUNK = 500  # spots per chunk — keeps peak RAM under 2 GB

    for region in unique_f:
        region_idx = [i for i, r in enumerate(regions_f) if r == region]
        n_spots    = len(region_idx)
        print(f'  Region "{region}": {n_spots} spots', end=' ... ')

        # Running stats for this region
        reg_sum  = {}  # gene -> sum
        reg_sq   = {}  # gene -> sum of squares
        reg_vals = {}  # gene -> sampled values (for percentiles)
        reg_n    = 0

        # Process in chunks of CHUNK spots
        for start in range(0, n_spots, CHUNK):
            chunk_idx = region_idx[start:start + CHUNK]
            X_chunk   = to_dense(fetal_vis.X[chunk_idx, :])
            chunk_n   = X_chunk.shape[0]

            for gi, gene in enumerate(gene_names_f):
                if gene not in TRACKED_GENES:
                    continue
                col = X_chunk[:, gi].astype(np.float64)

                # Region stats
                reg_sum[gene]  = reg_sum.get(gene, 0.0)  + col.sum()
                reg_sq[gene]   = reg_sq.get(gene, 0.0)   + (col ** 2).sum()
                if gene not in reg_vals:
                    reg_vals[gene] = []
                # Sample up to 200 values per region for percentile estimation
                if len(reg_vals[gene]) < 200:
                    reg_vals[gene].extend(col.tolist())

                # Overall stats
                overall_counts[gene] = overall_counts.get(gene, 0.0) + col.sum()
                overall_sq[gene]     = overall_sq.get(gene, 0.0)     + (col ** 2).sum()
                if gene not in overall_vals:
                    overall_vals[gene] = []
                if len(overall_vals[gene]) < 500:
                    overall_vals[gene].extend(col.tolist())

            reg_n       += chunk_n
            overall_n   += chunk_n
            del X_chunk
            gc.collect()

        # Finalise region baseline
        reg_baseline = {}
        reg_markers  = []
        for gene in reg_sum:
            mean = reg_sum[gene] / reg_n
            var  = max(0.0, reg_sq[gene] / reg_n - mean ** 2)
            vals = reg_vals.get(gene, [mean])
            reg_baseline[gene] = {
                'mean': round(mean, 4),
                'std':  round(float(np.sqrt(var)), 4),
                'p5':   round(float(np.percentile(vals, 5)), 4),
                'p95':  round(float(np.percentile(vals, 95)), 4),
            }

        # Cell type abundance for this region (from q05 columns — already in .obs)
        if q05_cols:
            region_obs = fetal_vis.obs.iloc[region_idx][q05_cols]
            ct_means   = region_obs.mean().sort_values(ascending=False)
            top_cts    = ct_means[ct_means > 0.05].head(8)
            ct_abundance_per_region[region] = [
                {'cell_type': c.replace('q05cell_abundance_w_sf_', ''),
                 'mean_abundance': round(float(v), 4)}
                for c, v in top_cts.items()
            ]

        region_baselines_f[region] = {
            'n_spots':        n_spots,
            'gene_baselines': reg_baseline,
        }
        print(f'done')

    # ── Build overall fetal baseline from running stats ───────────────────────
    overall_f = {}
    for gene in overall_counts:
        mean = overall_counts[gene] / overall_n
        var  = max(0.0, overall_sq[gene] / overall_n - mean ** 2)
        vals = overall_vals.get(gene, [mean])
        overall_f[gene] = {
            'mean': round(mean, 4),
            'std':  round(float(np.sqrt(var)), 4),
            'p5':   round(float(np.percentile(vals, 5)), 4),
            'p95':  round(float(np.percentile(vals, 95)), 4),
        }

    fetal_vis.file.close()
    del fetal_vis
    gc.collect()

    FETAL_RESULT = {
        'n_spots':                  overall_n,
        'region_column_used':       region_col_f,
        'n_regions':                len(unique_f),
        'region_names':             unique_f,
        'overall':                  overall_f,
        'regions':                  region_baselines_f,
        'ct_abundance_per_region':  ct_abundance_per_region,
        'n_slt_cell_types':         len(slt_cols),
        'n_q05_cell_types':         len(q05_cols),
    }

    mem = psutil.virtual_memory()
    print(f'\nFetal done. {len(overall_f)} gene baselines, {len(unique_f)} regions.')
    print(f'RAM after: {mem.available / 1e9:.1f} GB available')

else:
    print('Fetal Visium file not found — skipping.')


---
## Section 7 — Bronchus vs Parenchyma Comparison

The most important spatial fact in lung biology: the same gene behaves very
differently in airway vs. alveolar tissue. This section identifies the genes
with the strongest tissue-level contrast — these are the genes where spatial
context changes the interpretation of a deviation most dramatically.

In [ ]:
print('Computing bronchus vs. parenchyma contrast...')
print('These are genes where spatial context most changes the interpretation.\n')

# Find shared genes
shared_genes = set(overall_b.keys()) & set(overall_p.keys())

contrasts = []
for gene in shared_genes:
    b_mean = overall_b[gene]['mean']
    p_mean = overall_p[gene]['mean']
    delta  = b_mean - p_mean  # positive = higher in bronchus
    contrasts.append((gene, b_mean, p_mean, delta))

contrasts.sort(key=lambda x: abs(x[3]), reverse=True)

print(f'{'Gene':<12} {'Bronchus':>10} {'Parenchyma':>12} {'Δ (B-P)':>10}  Location')
print('-' * 70)
for gene, b, p, d in contrasts[:30]:
    location = 'AIRWAY' if d > 0 else 'ALVEOLAR'
    bar = '▶' * min(int(abs(d) * 3), 12)
    print(f'{gene:<12} {b:>10.3f} {p:>12.3f} {d:>+10.3f}  {location}  {bar}')

# Save contrast list as part of the artefact
TISSUE_CONTRASTS = [
    {
        'gene':              gene,
        'bronchus_mean':     round(b, 4),
        'parenchyma_mean':   round(p, 4),
        'log_fc':            round(d, 4),
        'enriched_in':       'bronchus' if d > 0 else 'parenchyma',
    }
    for gene, b, p, d in contrasts[:200]
]

print(f'\n{len(TISSUE_CONTRASTS)} genes with strong bronchus/parenchyma contrast saved.')

---
## Section 8 — Build Clinical Landmark Table

Map key clinical genes to their spatial expression pattern. This is what gets
shown in the deviation report: *"COL1A1 is elevated — this gene is normally
higher in bronchial submucosa than in alveolar parenchyma."*

In [ ]:
# Landmark genes — clinical relevance + spatial expectation
LANDMARKS = {
    # Fibrosis
    'COL1A1':  'Fibroblast collagen — marker of subpleural fibrosis when elevated in parenchyma',
    'COL3A1':  'Stromal collagen — early fibrotic remodelling',
    'ACTA2':   'Myofibroblast marker — fibrotic activation',
    'FN1':     'Fibronectin — extracellular matrix deposition',
    'TGFB1':   'TGF-β1 — master fibrosis driver',
    # Surfactant / AT2
    'SFTPC':   'Surfactant C — AT2 cell function marker (alveolar)',
    'SFTPB':   'Surfactant B — alveolar epithelial integrity',
    'ABCA3':   'Lipid transporter for surfactant production',
    # Airway
    'MUC5B':   'Mucin — airway goblet cells and glands (bronchial)',
    'MUC5AC':  'Mucin — airway epithelium',
    'SCGB1A1': 'Club cell marker — bronchiolar epithelium',
    'FOXJ1':   'Ciliated cell TF — mucociliary clearance',
    # Immune / infection
    'MX1':     'Interferon response — viral infection signal',
    'ISG15':   'Interferon-stimulated — innate antiviral',
    'S100A8':  'Myeloid/neutrophil — inflammatory infiltrate',
    'MARCO':   'Alveolar macrophage scavenger receptor',
    # Vascular
    'PECAM1':  'Endothelial — vascular integrity',
    'VEGFA':   'Angiogenesis — vascular remodelling',
    # Proliferation
    'MKI67':   'Proliferation — should be near-zero in healthy adult',
    'TOP2A':   'Cell cycle — abnormal if elevated in adult tissue',
    # TB-relevant
    'CXCL10':  'IFN-γ-induced — TB granuloma formation',
    'TNF':     'Cytokine — TB and inflammatory disease',
    'IL1B':    'Interleukin-1β — macrophage activation',
    'SPP1':    'Osteopontin — macrophage-driven fibrosis in TB',
}

print(f'{'Gene':<12} {'Bronchus':>10} {'Parenchyma':>12}  {'Δ':>7}  Location        Clinical note')
print('-' * 110)

LANDMARK_SPATIAL = {}
for gene, note in LANDMARKS.items():
    b_val = overall_b.get(gene, {}).get('mean')
    p_val = overall_p.get(gene, {}).get('mean')
    if b_val is None and p_val is None:
        print(f'{gene:<12}  not tracked in model')
        continue
    b_str = f'{b_val:.3f}' if b_val is not None else 'n/a'
    p_str = f'{p_val:.3f}' if p_val is not None else 'n/a'
    if b_val is not None and p_val is not None:
        d    = b_val - p_val
        loc  = 'AIRWAY   ' if d > 0.2 else ('ALVEOLAR ' if d < -0.2 else 'BOTH     ')
        d_str = f'{d:+.3f}'
    else:
        d_str = 'n/a'
        loc   = 'partial'
    print(f'{gene:<12} {b_str:>10} {p_str:>12}  {d_str:>7}  {loc}  {note[:50]}')
    LANDMARK_SPATIAL[gene] = {
        'bronchus_mean':   b_val,
        'parenchyma_mean': p_val,
        'note':            note,
    }

---
## Section 9 — Save Spatial Baselines Artefact

In [ ]:
spatial_baselines = {
    'built_at':           datetime.now(timezone.utc).isoformat(),
    'n_tracked_genes':    len(TRACKED_GENES),
    'bronchus':           BRONCHUS_RESULT,
    'parenchyma':         PARENCHYMA_RESULT,
    'tissue_contrasts':   TISSUE_CONTRASTS,
    'landmark_spatial':   LANDMARK_SPATIAL,
}

if FETAL_RESULT is not None:
    spatial_baselines['fetal'] = FETAL_RESULT

output_path = MODEL_DIR / 'spatial_baselines.json'
output_path.write_text(json.dumps(spatial_baselines, indent=2))

size_mb = output_path.stat().st_size / 1_000_000
print(f'Saved: {output_path}')
print(f'Size:  {size_mb:.1f} MB')
print()
print('Contents:')
print(f'  Bronchus:   {BRONCHUS_RESULT["n_spots"]} spots, {BRONCHUS_RESULT["n_regions"]} regions, {len(svgs_b)} SVGs')
print(f'  Parenchyma: {PARENCHYMA_RESULT["n_spots"]} spots, {PARENCHYMA_RESULT["n_regions"]} regions, {len(svgs_p)} SVGs')
if FETAL_RESULT:
    print(f'  Fetal:      {FETAL_RESULT["n_spots"]} spots, {FETAL_RESULT["n_regions"]} domains')
print(f'  Contrasts:  {len(TISSUE_CONTRASTS)} genes with bronchus/parenchyma contrast')
print(f'  Landmarks:  {len(LANDMARK_SPATIAL)} clinical landmark genes mapped spatially')

In [ ]:
# Update meta.json to record spatial baselines as built
meta = load_json('meta.json')
meta['has_spatial_baselines']  = True
meta['spatial_built_at']       = spatial_baselines['built_at']
meta['n_bronchus_spots']       = BRONCHUS_RESULT['n_spots']
meta['n_parenchyma_spots']     = PARENCHYMA_RESULT['n_spots']
meta['n_spatial_svgs']         = len(svgs_b) + len(svgs_p)
if FETAL_RESULT:
    meta['n_fetal_spatial_spots'] = FETAL_RESULT['n_spots']

(MODEL_DIR / 'meta.json').write_text(json.dumps(meta, indent=2))
print('meta.json updated:')
print(f'  has_spatial_baselines: {meta["has_spatial_baselines"]}')
print(f'  spatial_built_at:      {meta["spatial_built_at"]}')
print(f'  n_bronchus_spots:      {meta["n_bronchus_spots"]}')
print(f'  n_parenchyma_spots:    {meta["n_parenchyma_spots"]}')

---
## Section 10 — Verification

Confirm the artefact is complete, internally consistent, and biologically
sensible before loading it into the API.

In [ ]:
print('SPATIAL BASELINE VERIFICATION')
print('=' * 60)

# Reload from disk — verify it round-trips correctly
loaded = json.loads(output_path.read_text())

checks = []

# 1. File loads cleanly
checks.append(('File round-trips from JSON', True, f'{size_mb:.1f} MB'))

# 2. Both tissues present
ok = 'bronchus' in loaded and 'parenchyma' in loaded
checks.append(('Both tissue baselines present', ok, 'bronchus + parenchyma'))

# 3. Bronchus has expected gene count
n_b = len(loaded['bronchus']['overall'])
ok  = n_b > 1000
checks.append(('Bronchus has >1,000 gene baselines', ok, f'{n_b:,} genes'))

# 4. Parenchyma has expected gene count
n_p = len(loaded['parenchyma']['overall'])
ok  = n_p > 1000
checks.append(('Parenchyma has >1,000 gene baselines', ok, f'{n_p:,} genes'))

# 5. SFTPC is higher in parenchyma than bronchus (alveolar gene)
sftpc_b = loaded['bronchus']['overall'].get('SFTPC', {}).get('mean', 0)
sftpc_p = loaded['parenchyma']['overall'].get('SFTPC', {}).get('mean', 0)
ok = sftpc_p > sftpc_b
checks.append(('SFTPC higher in parenchyma than bronchus', ok,
               f'bronchus={sftpc_b:.3f} parenchyma={sftpc_p:.3f}'))

# 6. MUC5B is higher in bronchus than parenchyma (airway mucin gene)
muc5b_b = loaded['bronchus']['overall'].get('MUC5B', {}).get('mean', 0)
muc5b_p = loaded['parenchyma']['overall'].get('MUC5B', {}).get('mean', 0)
ok = muc5b_b >= muc5b_p  # should be higher in airway
checks.append(('MUC5B higher in bronchus than parenchyma', ok,
               f'bronchus={muc5b_b:.3f} parenchyma={muc5b_p:.3f}'))

# 7. SVGs are non-empty
n_svgs = len(loaded['bronchus']['spatially_variable_genes']) + len(loaded['parenchyma']['spatially_variable_genes'])
ok = n_svgs > 0
checks.append(('Spatially variable genes identified', ok, f'{n_svgs} total'))

# 8. meta.json updated
meta_check = json.loads((MODEL_DIR / 'meta.json').read_text())
ok = meta_check.get('has_spatial_baselines') is True
checks.append(('meta.json updated: has_spatial_baselines=true', ok, str(ok)))

# 9. Tissue contrast list non-empty
n_contrast = len(loaded.get('tissue_contrasts', []))
ok = n_contrast > 10
checks.append(('Tissue contrast list non-empty', ok, f'{n_contrast} gene contrasts'))

# 10. Clinical landmarks mapped
n_lm = len(loaded.get('landmark_spatial', {}))
ok   = n_lm >= 5
checks.append(('Clinical landmark genes mapped', ok, f'{n_lm} genes'))

# Print results
passed = 0
for label, ok, detail in checks:
    status = '✓ PASS' if ok else '✗ FAIL'
    passed += int(ok)
    print(f'  {status}  {label:<50}  {detail}')

print()
print(f'Result: {passed}/{len(checks)} checks passed')

if passed == len(checks):
    print()
    print('Spatial baselines are valid and biologically coherent.')
    print('The deviation engine will now include tissue location context in every report.')
else:
    print()
    print('Some checks failed. Review the failed checks above before deploying.')

---
## Section 11 — What the Engine Now Reports

Preview how spatial context changes a deviation finding. Same gene,
richer interpretation.

In [ ]:
print('EXAMPLE: HOW SPATIAL CONTEXT CHANGES A DEVIATION REPORT')
print('=' * 70)
print()

examples = [
    ('COL1A1', 4.21, 'elevated'),
    ('SFTPC',  0.20, 'suppressed'),
    ('MUC5B',  3.10, 'elevated'),
    ('MX1',    3.80, 'elevated'),
]

for gene, sample_val, direction in examples:
    b = loaded['bronchus']['overall'].get(gene, {})
    p = loaded['parenchyma']['overall'].get(gene, {})
    b_mean = b.get('mean', None)
    p_mean = p.get('mean', None)

    print(f'Gene: {gene}  (sample value: {sample_val}, direction: {direction})')

    if b_mean is not None and p_mean is not None:
        d = b_mean - p_mean
        if abs(d) > 0.2:
            dominant = 'bronchus (airway)' if d > 0 else 'parenchyma (alveolar)'
            print(f'  Healthy baseline:  bronchus {b_mean:.3f} | parenchyma {p_mean:.3f}')
            print(f'  Spatial pattern:   normally higher in {dominant}')
            if direction == 'elevated':
                if d < 0:  # alveolar gene that is elevated
                    print(f'  Interpretation:   Elevation is UNEXPECTED — this gene is\'s alveolar, so')
                    print(f'                    elevation suggests disruption of alveolar biology.')
                else:
                    print(f'  Interpretation:   Gene is normally dominant in airway.')
                    print(f'                    Elevation suggests airway-specific process.')
            elif direction == 'suppressed':
                if d < 0:  # alveolar gene suppressed
                    print(f'  Interpretation:   Suppression of an alveolar gene — AT2/alveolar')
                    print(f'                    function is compromised.')
        else:
            print(f'  Spatial pattern: expressed similarly in both tissues (Δ={d:+.3f})')
            print(f'  Bronchus: {b_mean:.3f} | Parenchyma: {p_mean:.3f}')

    landmark = loaded.get('landmark_spatial', {}).get(gene, {})
    if landmark.get('note'):
        print(f'  Clinical note:    {landmark["note"]}')
    print()

---
## Section 12 — Next Steps

### Immediately after this notebook

1. **Download `spatial_baselines.json`** from Google Drive
   (`HEALTH/models/respiratory_model/spatial_baselines.json`)
   and copy it into the local repo under `models/respiratory_model/`.

2. **Restart the FastAPI server** and confirm:
   ```bash
   curl http://localhost:8000/api/v1/model/status
   # → has_spatial_baselines: true
   ```

3. **Update `app/services/anomaly_detector.py`** to load `spatial_baselines.json`
   and add spatial context to deviation report outputs.
   The deviation report will then include:
   - `spatial_context`: which tissue is most affected by each gene deviation
   - `tissue_enrichment`: bronchus vs. parenchyma baseline for the deviated gene

### What this closes

Before this notebook, the engine answered:
- *What is biologically wrong?* ✓

After this notebook, the engine answers:
- *What is biologically wrong?* ✓
- *Where in the lung tissue is it wrong?* ✓ ← new

This directly addresses the limitation raised about intracellular/subregional
infections: while the engine cannot see inside a single cell, it can now tell
you whether the deviation is concentrated in airway tissue vs. alveolar tissue,
which is the clinically meaningful spatial distinction for diseases like TB
(bronchocentric) vs. IPF (subpleural parenchymal) vs. COVID-19 (diffuse alveolar).

### Notebook 06 (future)

- Load ATAC-seq data (`lung_fetal_atac.h5ad`) and build chromatin accessibility
  baselines — detects early epigenetic changes before gene expression changes
- Requires adult ATAC-seq data (the current file is fetal); KEMRI partnership
  could provide adult lung ATAC-seq from TB/COPD patients